# Offline Design Experiment Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/instadeepai/alf/blob/main/tutorials/experiments/offline_design_tutorial.ipynb)

_Open in Colab works once ALF is public / on PyPI; until then, use the local setup below._

This tutorial demonstrates how to run an offline design experiment using the ALF (Active Learning Framework) codebase. We'll walk through the complete process of setting up and running an offline active learning experiment, and our use case in this example notebook is: protein design using the GFP (Green Fluorescent Protein) dataset.

### What is an Offline Design Experiment?

An offline design experiment is a type of active learning experiment where we fixed labelled dataset to provide us with groundtruth labels instead of collecting them in real time. The key advantage of offline experiments is that we can simulate the entire process using existing data, making it perfect for:
- Algorithm development and testing
- Benchmarking different acquisition strategies
- Understanding how active learning performs on real world data

### Experiment Overview

In this tutorial, we'll touch on each part of an offline design experiment:
- Set up the GFP dataset with protein sequences and their fitness values
- Initialise a CNN surrogate model to predict protein fitness
- Use a greedy acquisition strategy to select promising sequences
- Run multiple rounds of active learning
- Analyse the results to see how well we can discover high-fitness proteins

These are presented as components rather than a strict sequence of steps: the surrogate is not pre-trained separately here — `DesignTask` trains it on the labelled data as part of each acquisition round.

### Framework Components

Before we start, let's understand the key components of the ALF framework:

1. **Dataset** ([`GFP`](https://instadeepai.github.io/alf/api/alf_tools/datasets/gfp/)): Contains the data and their labels and handles data splitting into train/validation/test/candidate_pool sets.

2. **Surrogate Model** ([`CNNModel`](https://instadeepai.github.io/alf/api/alf_tools/models/cnn/)): Trained on labelled data to approximate the expensive experimental evaluation.

3. **Search Strategy** ([`DatasetSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/)): Defines the search space (all possible sequences to choose from). In offline experiments, this is typically the unlabelled portion of the dataset.

4. **Acquisition Function** ([`AcquisitionFunction`](https://instadeepai.github.io/alf/api/alf_core/optimizer/acquisition_function/)): Determines which sequences are most promising to evaluate next, in this case, we use "greedy" selection (pick the highest predicted fitness).

5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): Combines acquisition function and search strategy and handles the "ask" (select candidates) and "tell" (update with results) cycle.

6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/)): Simulates the expensive experimental evaluation, in offline experiments, provides ground truth labels from the dataset.

7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): Orchestrates the entire active learning loop, and manages multiple acquisition rounds and model updates.

### Step 0: Setup

Run the cell below to install ALF and this tutorial's dependencies — **no repository clone required**, so it works in a fresh environment or on Google Colab.

- Already set up a dev environment from a clone (`uv sync`)? You can **skip the install cell**.
- To run on a **GPU**, uncomment the GPU line in the install cell.

For all installation options, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).
- On **Colab**, the first install can take a few minutes; if prompted, choose *Runtime ▸ Restart session* and re-run the cell. To use a GPU, set *Runtime ▸ Change runtime type ▸ GPU* and uncomment the GPU line in the install cell.

In [ ]:
# Install ALF + this tutorial's dependencies — no clone needed.
# (Skip this cell if you are already running from a cloned repo via `uv sync`.)
# TODO(pypi): once ALF is published to PyPI, replace the git install below with:
#   %pip install alf_tools matplotlib pandas
%pip install "alf_core @ git+https://github.com/instadeepai/alf.git#subdirectory=core" "git+https://github.com/instadeepai/alf.git#subdirectory=tools" matplotlib pandas
# GPU (optional): run this AFTER the line above to switch PyTorch to a CUDA build.
# %pip install torch --index-url https://download.pytorch.org/whl/cu128

### Step 1: Import Required Libraries

Let's start by importing all the necessary components from the ALF framework:


In [ ]:
# Core framework imports
import shutil

import matplotlib.pyplot as plt

# Additional imports for analysis
import pandas as pd
from alf_core import (
    DatasetSearch,
    DesignTask,
    FileStateLogger,
    Optimizer,
    Oracle,
    Surrogate,
    TerminalStateLogger,
)
from alf_tools.datasets import GFP
from alf_tools.models import CNNModel
from alf_tools.optimizer.acquisition_functions import Greedy

print("✅ All imports successful!")

### Step 2: Configure the Dataset

The GFP dataset contains protein sequences and their measured brightness values. Let's set up the dataset with appropriate splits using [`BaseDatasetConfig`](https://instadeepai.github.io/alf/api/alf_core/dataset/base_dataset/) and [`Modality`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/):

In [ ]:
# Dataset configuration
from alf_core.dataclasses.candidate import Modality
from alf_core.dataset.base_dataset import BaseDatasetConfig
from alf_core.utils.enums import ProblemType

# Initialise the GFP dataset
gfp_dataset = GFP(
    BaseDatasetConfig(
        name="gfp",
        modality=Modality.SEQUENCE,  # We're working with protein sequences
        seed=51505,  # For reproducibility
        train_ratio=0.1,  # 10% for initial training
        validation_frac=0.1,  # 10% of training data for validation
        test_ratio=0.1,  # 10% for final evaluation
        split_type="random",
        problem_type=ProblemType.REGRESSION,  # GFP brightness is a continuous score
    )
)

print("✅ Dataset initialised!")
print("📊 Dataset info:")
print(f"   - Total sequences: {len(gfp_dataset._raw_dataset)}")
print(f"   - Sequence length: {len(gfp_dataset._raw_dataset.data[0])}")
print(
    f"   - Fitness score range: {gfp_dataset._raw_dataset.labels.min():.2f} to"
    f" {gfp_dataset._raw_dataset.labels.max():.2f}"
)

### Step 3: Initialise the Surrogate Model

The surrogate model is a CNN that learns to predict protein fitness from sequence. We wrap it in a [`Surrogate`](https://instadeepai.github.io/alf/api/alf_core/surrogate/) - this is the "brain" of our active learning system:

In [ ]:
# Initialise the CNN surrogate model
surrogate = Surrogate(model=CNNModel())

print("✅ Surrogate model initialised!")
print("🧠 Model architecture:")
print("   - Type: 1D Convolutional Neural Network")
print("   - Input: One-hot encoded protein sequences")
print("   - Output: Predicted fitness values")
print("   - Purpose: Learn sequence → fitness mapping")

### Step 4: Set Up the Acquisition Strategy

The acquisition function determines which sequences are most promising to evaluate next. We'll use a simple [`Greedy`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/greedy/) strategy, which always picks the candidates with the highest predicted fitness.

Greedy is used here for illustration because it is easy to reason about, but it ignores the surrogate's uncertainty and so tends to over-exploit. In practice, acquisition functions that balance exploration and exploitation usually perform better — for example [`UCB`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/ucb/) (upper confidence bound) or [`ExpectedImprovement`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/expected_improvement/).

In [ ]:
# Initialise acquisition function (greedy strategy)
acquisition_fn = Greedy()

print("✅ Acquisition function initialised!")
print("🎯 Strategy: Greedy selection")
print("   - Selects sequences with highest predicted fitness")
print("   - Simple but effective for many design tasks")
print("   - Alternative strategies: UCB, Thompson sampling, etc.")

### Step 5: Configure the Search Strategy

The search strategy defines where we can look for new sequences. In offline experiments, we search within the unlabelled portion of our dataset:

In [ ]:
# Initialise search strategy
search_fn = DatasetSearch()

print("✅ Search strategy initialised!")
print("🔍 Search space: Candidate pool of GFP dataset")
print("   - Starts with sequences not in train/val/test splits")
print("   - Gradually shrinks as we acquire more labels")
print("   - In online experiments, this could be a generative model or combinatorial space")

### Step 6: Create the Optimizer

The optimizer combines the acquisition function and search strategy to handle the ask-tell cycle:


In [ ]:
# Initialise optimizer
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

print("✅ Optimizer initialised!")
print("⚙️ Optimizer components:")
print("   - Acquisition function: Greedy selection")
print("   - Search strategy: Dataset-based search")
print("   - Handles: ask() → select candidates, tell() → update surrogate with results")

### Step 7: Set Up the Oracle

The oracle simulates the expensive experimental evaluation. In offline experiments, it provides ground truth labels from the dataset:


In [ ]:
# Initialise oracle
oracle = Oracle(scorer=gfp_dataset)

print("✅ Oracle initialised!")
print("🔮 Oracle function:")
print("   - Simulates expensive experimental evaluation")
print("   - Returns ground truth fitness values from dataset")
print("   - In real experiments, this would be actual lab measurements")

### Step 8: Configure the Design Task

Now we'll set up the main design task that orchestrates the entire active learning process:


In [ ]:
# Configure the design task
num_acq_rounds = 5  # Number of active learning rounds
acq_batch_size = 100  # Number of sequences to acquire per round

task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)

print("✅ Design task configured!")
print("📋 Experiment parameters:")
print(f"   - Acquisition rounds: {num_acq_rounds}")
print(f"   - Batch size per round: {acq_batch_size}")
print(f"   - Total sequences to acquire: {num_acq_rounds * acq_batch_size}")
print("   - Task type: Offline design optimisation")

### Step 9: Run the Offline Design Experiment

Now let's run the complete experiment! This will:
1. Set up the initial state with training data
2. Run multiple rounds of active learning
3. Track performance metrics throughout


In [ ]:
import logging
from pathlib import Path

# Keep the output readable: only surface warnings and above from the framework.
# Raise this to logging.INFO if you want the full per-round trace.
logging.basicConfig(level=logging.WARNING)

# Initialise logger for tracking metrics
terminal_logger = TerminalStateLogger()
save_path = Path("results/offline_design/")
if save_path.exists():
    shutil.rmtree(save_path)
file_logger = FileStateLogger(output_path=save_path)
loggers = [terminal_logger, file_logger]

# Set up the initial state and run the experiment
print("🚀 Setting up experiment...")
state = task.setup(dataset=gfp_dataset, surrogate=surrogate)
print(
    f"   train={len(state.dataset.train_dataset)}, "
    f"val={len(state.dataset.validation_dataset)}, "
    f"test={len(state.dataset.test_dataset)}, "
    f"pool={len(state.dataset.candidate_pool)}"
)

print("🔄 Running offline active learning experiment...")
task.run(state, state_loggers=loggers, optimizer=optimizer, oracle=oracle)
print("✅ Experiment completed!")

### Step 10: Analyse the Results

After the experiment completes, we can load the saved metrics and visualise the optimisation progress. We track:

- **Round Mean Fitness:** Average fitness of selected sequences in each acquisition batch
- **Top 10% Recall:** Percentage of sequences in the acquired batch that are in the top 10% of all sequences by fitness
- **Round Maximum Fitness:** Highest fitness value among sequences selected in each round
- **Surrogate Spearman:** Spearman rank correlation coefficient between the surrogate model's predictions and true fitness values, measuring the strength of the monotonic relationship (ranges from -1 to 1, where 1 indicates perfect rank agreement)

If the optimisation is working well, the first three metrics should increase over rounds as we discover better sequences, while the Spearman correlation should remain high, indicating the surrogate model maintains good predictive quality.

> These are **per-round** metrics, stored one row per round in `metrics.csv`. Per-experiment aggregate metrics (e.g. `auc_top_k`) are written separately to `summary.csv` in the same results directory.

In [ ]:
# Load metrics from CSV
metrics = pd.read_csv(save_path / "metrics.csv")

# Keep only the acquisition rounds. Dropping rows with no acquired-batch metrics removes the
# initial round-0 row, and also any per-experiment summary row that some logger versions append
# to metrics.csv (newer versions write summary metrics to a separate summary.csv instead). This
# keeps the analysis correct regardless of the logger version.
acq = metrics.dropna(subset=["acquired_candidates/round_mean"])
if acq.empty:
    raise ValueError("metrics.csv has no acquisition rounds — did the experiment run?")

print("Experiment Summary:")
print(f"Acquisition rounds: {len(acq)}")
print(f"Initial Train Set Mean Fitness: {metrics['dataset/train_mean'].iloc[0]:.4f}")
print(f"Final Mean Fitness: {acq['acquired_candidates/round_mean'].iloc[-1]:.4f}")
print(f"Best Fitness Found: {acq['acquired_candidates/round_max'].max():.4f}")
print(f"Final Spearman Correlation: {acq['surrogate/test_spearman'].iloc[-1]:.4f}")

With the metrics loaded, we plot the four tracked quantities across the acquisition rounds so we can see the optimisation progress at a glance:

In [ ]:
# Create visualisation of results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Protein Design Optimisation Results", fontsize=16, fontweight="bold")

# `acq` (acquisition rounds only) was computed above. Use the explicit `round` column for the
# x-axis when the logger provides one, otherwise fall back to the row index.
rounds = acq["round"] if "round" in acq.columns else acq.index

# 1. Round Mean Fitness
axes[0, 0].plot(
    rounds,
    acq["acquired_candidates/round_mean"],
    marker="o",
    linewidth=2,
    markersize=8,
    color="#e74c3c",
)
axes[0, 0].set_xlabel("Round", fontsize=12)
axes[0, 0].set_ylabel("Mean Fitness", fontsize=12)
axes[0, 0].set_title("Round Mean Fitness", fontsize=13, fontweight="bold")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticks(rounds)

# 2. Top 10% Recall (Batch)
axes[0, 1].plot(
    rounds, acq["optimizer/top_10pc_recall"], marker="^", linewidth=2, markersize=8, color="#2ecc71"
)
axes[0, 1].set_xlabel("Round", fontsize=12)
axes[0, 1].set_ylabel("Top 10% Recall", fontsize=12)
axes[0, 1].set_title("Top 10% Recall in Batch", fontsize=13, fontweight="bold")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])
axes[0, 1].set_xticks(rounds)

# 3. Maximum Fitness per Round
axes[1, 0].plot(
    rounds,
    acq["acquired_candidates/round_max"],
    marker="D",
    linewidth=2,
    markersize=8,
    color="#9b59b6",
)
axes[1, 0].set_xlabel("Round", fontsize=12)
axes[1, 0].set_ylabel("Maximum Fitness", fontsize=12)
axes[1, 0].set_title("Round Maximum Fitness", fontsize=13, fontweight="bold")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xticks(rounds)

# 4. Surrogate Spearman Correlation
axes[1, 1].plot(
    rounds, acq["surrogate/test_spearman"], marker="v", linewidth=2, markersize=8, color="#34495e"
)
axes[1, 1].set_xlabel("Round", fontsize=12)
axes[1, 1].set_ylabel("Spearman Correlation", fontsize=12)
axes[1, 1].set_title("Surrogate Prediction Quality", fontsize=13, fontweight="bold")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(rounds)

plt.tight_layout()
plt.show()

#### Interpreting the results

Reading the four panels together tells us whether the active learning loop is doing its job:

- **Round Mean Fitness** and **Round Maximum Fitness** show the quality of the batch acquired each round. With a greedy strategy on GFP, the surrogate quickly learns to point at the brighter region of sequence space, so these typically rise over the first rounds before plateauing as the best candidates are exhausted from the pool.
- **Top 10% Recall** measures how many of the genuinely top-decile sequences the loop manages to pull into each acquired batch. Higher is better; values well above the ~10% you'd expect from random sampling indicate the surrogate is successfully prioritising high-fitness candidates.
- **Surrogate Spearman** tracks how well the model's predicted ranking matches the true ranking on the held-out test set. It should stay reasonably high and stable — if it collapses, the surrogate is no longer a trustworthy guide and the acquisition decisions degrade with it.

Exact numbers vary run to run (different seeds, the stochastic CNN training, and the small per-round batch), so focus on the *trends* rather than any single value. A healthy run shows fitness climbing while Spearman holds up; a flat or noisy recall curve is a hint that a more exploratory acquisition function (e.g. `UCB` or `ExpectedImprovement`) might find better sequences.

In [ ]:
# Optionally, clean up the results directory
if save_path.exists():
    shutil.rmtree(save_path)

print("✅ Results directory cleaned up!")

## Conclusion

This tutorial has demonstrated how to run an **offline design experiment** using the ALF framework. We've learned:

- **What offline design experiments are** and why they're valuable for algorithm development
- **How to set up all the necessary components** (dataset, surrogate model, acquisition function, search strategy, oracle)
- **How to run a complete active learning experiment** with multiple acquisition rounds
- **How to analyse and interpret the results** to understand the experimental process

The ALF framework provides a flexible and modular approach to active learning that can be adapted to many different design problems. Whether you're working on protein design, drug discovery, materials science, or other optimisation tasks, the principles demonstrated here can be applied to guide efficient experimental design.

**Happy designing!** 🧬🔬✨
